# Chapter 19 · Hidden neurons invent new clues
**Deck 4 · Live demo.** Build a 2-2-1 network by hand: two hidden clues (“both evening?”, “both morning?”) and an output (“either?”).

**You will:** write one neuron · build the two clues · combine them · prove 4 of 4.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "font.size": 12})

## 1 · Your prediction
Which student turns on hidden neuron 2 (“both morning?”)?

In [ ]:
my_prediction = "Kabir"

In [ ]:
# The body-clock students: x1 = exam time (0 morning, 1 evening), x2 = body clock (0 early bird, 1 night owl)
names = ['Kabir', 'Zara', 'Tara', 'Omar']
X = np.array([[0, 0], [1, 1], [1, 0], [0, 1]], dtype=float)
y = np.array([1, 1, 0, 0], dtype=float)          # pass when exam time matches body clock
for n, (a, b), r in zip(names, X, y):
    print(f"{n:6} exam {'evening' if a else 'morning':8} {'night owl' if b else 'early bird':10} → {'pass' if r else 'fail'}")

## 2 · One neuron: weighted sum, then a switch

In [ ]:
def neuron(x, w, b):
    z = x @ np.array(w) + b
    return z, (z > 0).astype(float)

## 3 · The two invented clues and the output

In [ ]:
z1, h1 = neuron(X, [1, 1], -1.5)      # both evening?
z2, h2 = neuron(X, [-1, -1], 0.5)     # both morning?
H = np.c_[h1, h2]
z3, out = neuron(H, [1, 1], -0.5)     # either clue?
for n, a, b, c, d, o in zip(names, z1, h1, z2, h2, out):
    print(f'{n:6} hidden1 z {a:+.1f} → {b:.0f} · hidden2 z {c:+.1f} → {d:.0f} · output {o:.0f}')

## 3b · Many weights work
Try every w₁, w₂, b on a grid and count how many make a perfect “both evening?” neuron.

In [ ]:
grid = np.arange(-2, 2.01, 0.5)
want = np.array([0, 1, 0, 0])
good = [(a, b, c) for a in grid for b in grid for c in np.arange(-3, 3.01, 0.5)
        if np.array_equal(neuron(X, [a, b], c)[1], want)]
print(f'{len(good)} different (w1, w2, b) make a perfect “both evening?” neuron, e.g.', good[:4])
assert (1.0, 1.0, -1.5) in good and len(good) > 10
print('✓ no single right answer: many weights work')

## 3c · How many hidden neurons? Let training decide
Train sigmoid networks with 1, 2, 3, 4 and 8 hidden neurons, 50 random starts each (3,000 epochs, η = 1), and count the runs that get all four students right.

In [ ]:
sig = lambda z: 1 / (1 + np.exp(-z))
Y = y.astype(float)

def train(H, seed, epochs=3000, eta=1.0):
    r = np.random.default_rng(seed)
    W1, b1, W2, b2 = r.normal(0, 1, (2, H)), np.zeros(H), r.normal(0, 1, H), 0.0
    for _ in range(epochs):
        Hh = sig(X @ W1 + b1); P = sig(Hh @ W2 + b2); d2 = (P - Y) / 4
        dH = np.outer(d2, W2) * Hh * (1 - Hh)
        W2 -= eta * Hh.T @ d2; b2 -= eta * d2.sum(); W1 -= eta * X.T @ dH; b1 -= eta * dH.sum(0)
    P = sig(sig(X @ W1 + b1) @ W2 + b2)
    return int(((P >= 0.5) == (Y == 1)).sum())

sweep = {H: sum(train(H, s) == 4 for s in range(50)) for H in (1, 2, 3, 4, 8)}
for H, ok in sweep.items():
    print(f'{H} hidden: {ok:2d} of 50 runs learned all four')
assert sweep == {1: 0, 2: 39, 3: 46, 4: 50, 8: 50}, 'same table as the slide'
print('✓ too few can’t learn; a few spare make it reliable')

## 4 · PROOF

In [ ]:
assert list(h1) == [0, 1, 0, 0] and list(h2) == [1, 0, 0, 0], 'each clue fires for one student'
assert list(out) == list(y), 'all four right'
assert names[int(np.argmax(h2))] == my_prediction
_, h1a = neuron(np.array([[0., 0.]]), [1, 1], -1.5); _, h2a = neuron(np.array([[0., 0.]]), [-1, -1], 0.5)
assert neuron(np.c_[h1a, h2a], [1, 1], -0.5)[1][0] == 1, 'Asha passes'
print('✓ proof: 4 of 4, Asha passes')

## 5 · Your turn
Build a network for “pass when they DON’T match” (y = [0, 0, 1, 1]).

In [ ]:
# TODO

## Recap
Hidden neurons turn raw inputs into new clues. With the right clues, the output needs only one straight cut.